# Task 1: Repository and Kaggle setup

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. Run with **Save Version > Save & Run All (Commit)**.

This notebook clones the repository, installs the dependencies without reinstalling Kaggle's PyTorch,
confirms that two T4 GPUs are visible, and records the package versions in
`outputs/setup/setup_report.json`, archived as `task1_setup_outputs.zip`.

In [ ]:
REPO_URL = "https://github.com/mabdullahi7780/small-model-alignment-starter.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/small-model-alignment-starter"
OUT_DIR = f"{WORK_DIR}/outputs/setup"
EXPECTED_GPUS = 2

In [ ]:
import os
import subprocess
import sys


def run(cmd, cwd=None, env=None, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)


if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=commit %H%nauthor date %ad%nsubject %s"], cwd=REPO_DIR)

In [ ]:
# Kaggle's preinstalled torch is built for its GPU runtime; pin it so pip can never replace it.
TORCH_BEFORE = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Preinstalled torch:", TORCH_BEFORE)

constraint_file = f"{WORK_DIR}/torch-constraint.txt"
with open(constraint_file, "w") as handle:
    handle.write(f"torch=={TORCH_BEFORE}\n")

run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-c", constraint_file], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=REPO_DIR)

## GPUs as the driver sees them

In [ ]:
run(["nvidia-smi"])

## Versions, GPUs and checks

Runs in a fresh interpreter so it sees the packages installed above. Fails the notebook if a check does not pass.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
run(
    [
        sys.executable, "-m", "pa2.env_report",
        "--repo-dir", REPO_DIR,
        "--torch-before", TORCH_BEFORE,
        "--expect-gpus", str(EXPECTED_GPUS),
        "--out", f"{OUT_DIR}/setup_report.json",
    ],
    cwd=REPO_DIR,
)
with open(f"{OUT_DIR}/pip_freeze.txt", "w") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)

## Test harness check

Only the signature test is expected to pass at this stage; the adapter functions are implemented in Tasks 3 and 4.

In [ ]:
run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_public_adapter_smoke.py::test_required_functions_have_the_documented_signatures",
        "tests/test_env_report.py",
    ],
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": REPO_DIR},
)

## Archive the outputs

In [ ]:
import shutil

archive = shutil.make_archive(f"{WORK_DIR}/task1_setup_outputs", "zip", OUT_DIR)
print("Saved", archive)
print(sorted(os.listdir(OUT_DIR)))